# Generate Tasks
This notebook demonstrates how to create the tasks for the raster processing of the annual summaries and the vector processing of the annual summaries.

## Getting started

### Load required python packages

In [1]:
import json
import logging
from collections import defaultdict

import click
import rioxarray
from tqdm import tqdm

from water_quality.io import (
    check_directory_exists,
    find_geotiff_files,
    get_filesystem,
    join_url,
)
from water_quality.logs import setup_logging
from water_quality.io import get_basename

### Define analysis parameters

The expected inputs are:

- `historical_extent_rasters_dir`: This is the path to the directory containing the tiled COGs of the rasterized DE Africa Waterbodies Historical Extent polygons. Note that this rasters should have a resolution of 10m . See the cli tool [`wqms-summaries generate-rasters`](https://github.com/vikineema/deafrica_water_quality/blob/develop/src/water_quality/cli/summaries/rasterise_polygons.py).
- `output_dir`: The directory to write the text file containing the historical extent rasters file paths for raster processing and the json file containing the waterbodies uids and their corresponding COG paths for vector processing. Write the tasks to the `data` directory. These tasks should then be used in the continental workflows in Argo instead of running the cli tool `wqms-summaries generate-tasks` in Argo.

In [2]:
historical_extent_rasters_dir = "s3://deafrica-water-quality-dev/historical-extent-rasters/"
output_dir = "../../src/water_quality/data/"
log="INFO"

In [3]:
log_level = getattr(logging, log.upper())
_log = setup_logging(log_level)

In [4]:
fs = get_filesystem(path=output_dir, anon=False)
if not check_directory_exists(path=output_dir):
    fs.mkdirs(path=output_dir, exist_ok=True)

## Generate tasks

In [5]:
_log.info(
    f"Searching for waterbodies historical extent cogs in {historical_extent_rasters_dir} ..."
)
historical_extent_cogs = find_geotiff_files(historical_extent_rasters_dir)

cog_path_to_uids = {}
for cog_path in tqdm(historical_extent_cogs):
    ds = rioxarray.open_rasterio(cog_path)
    wbid_to_uid = json.loads(ds.attrs["WB_ID_to_UID"])
    uids = sorted(list(wbid_to_uid.values()))
    cog_path_to_uids[cog_path] = uids

# Sort to ensure during parallel processing
# tasks requiring the shortest time are processed first.
cog_path_to_uids = dict(
    sorted(cog_path_to_uids.items(), key=lambda x: len(x[1]))
)

assert len(cog_path_to_uids) == len(historical_extent_cogs)

_log.info(
    f"Found {len(historical_extent_cogs)} waterbodies historical extent cogs"
)

2026-03-22 13:05:12,172 __main__ [INFO]: Searching for waterbodies historical extent cogs in s3://deafrica-water-quality-dev/historical-extent-rasters/ ...
100%|██████████| 2779/2779 [03:56<00:00, 11.76it/s]
2026-03-22 13:09:09,196 __main__ [INFO]: Found 2779 waterbodies historical extent cogs


In [6]:
for key, value in list(cog_path_to_uids.items())[:3]:
    print(key, value)

s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x165y108.tif ['esm72v920u']
s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x166y106.tif ['eeypdwqrdk']
s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x166y109.tif ['esmyd1sh84']


In [7]:
raster_processing_tasks = list(cog_path_to_uids.keys())

raster_processing_tasks_fp = join_url(
    output_dir, "raster_processing_tasks.txt"
)

with fs.open(raster_processing_tasks_fp, "w") as file:
    file.write("\n".join(raster_processing_tasks))
_log.info(
    f"Historical extent rasters for raster processing of water quality summaries written to {raster_processing_tasks_fp}"
)

2026-03-22 13:09:09,206 __main__ [INFO]: Historical extent rasters for raster processing of water quality summaries written to ../../src/water_quality/data/raster_processing_tasks.txt


In [8]:
_log.info(
    "Searching for waterbodies covered by more than 1 historical extent COG ..."
)

uids_to_cog_paths = defaultdict(list)
for cog_path, uids in cog_path_to_uids.items():
    for uid in uids:
        uids_to_cog_paths[uid].append(cog_path)

# Sort to ensure during parallel processing
# tasks requiring the shortest time are processed first.
uids_to_cog_paths = dict(
    sorted(uids_to_cog_paths.items(), key=lambda x: len(x[1]))
)

2026-03-22 13:09:09,211 __main__ [INFO]: Searching for waterbodies covered by more than 1 historical extent COG ...


In [9]:
for key, value in list(uids_to_cog_paths.items())[:3]:
    print(key, value)

esm72v920u ['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x165y108.tif']
eeypdwqrdk ['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x166y106.tif']
esmyd1sh84 ['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x166y109.tif']


In [10]:
multi_tile_uids = {}
for uid, cog_paths in uids_to_cog_paths.items():
    if len(cog_paths) > 1:
        multi_tile_uids[uid] = cog_paths

# Sort to ensure during parallel processing
# tasks requiring the shortest time are processed first.
multi_tile_uids = dict(
    sorted(multi_tile_uids.items(), key=lambda x: len(x[1]))
)

In [11]:
for key, value in list(multi_tile_uids.items())[:3]:
    print(key, value)

evt1t6hr8f ['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x177y116.tif', 's3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x176y116.tif']
smds11f3dr ['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x195y116.tif', 's3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x195y117.tif']
sg5s70brp4 ['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x220y100.tif', 's3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x219y100.tif']


In [12]:
_log.info(
    f"Found {len(multi_tile_uids)} waterbodies covered by more than 1 historical extent COG. "
    "These waterbodies will need to be processed seperately during generation of water quality summaries."
)

2026-03-22 13:09:10,695 __main__ [INFO]: Found 6105 waterbodies covered by more than 1 historical extent COG. These waterbodies will need to be processed seperately during generation of water quality summaries.


In [13]:
# Previous continental runs show that waterbodies covered by the following
# Historical Extent COGs meet the criteria of only being covered by a
# single tile but have difficulty being processed using the raster-processing
# tool and thus must be procesed using the vector processing tool.
cog_paths_exceptions_filter = [
    "historical_extent_x177y097.tif",
    "historical_extent_x191y093.tif",
    "historical_extent_x208y056.tif",
    "historical_extent_x214y047.tif",
]
cog_paths_exceptions = [
    i
    for i in historical_extent_cogs
    if get_basename(i) in cog_paths_exceptions_filter
]

_log.info(
    f"Identifying waterbodies in {len(cog_paths_exceptions)} historical extent COGs "
    "that cannot be processed using the raster processing tool"
)

2026-03-22 13:09:10,749 __main__ [INFO]: Identifying waterbodies in 4 historical extent COGs that cannot be processed using the raster processing tool


In [14]:
uids_exceptions = []
for cog_path in cog_paths_exceptions:
    ds = rioxarray.open_rasterio(cog_path)
    wbid_to_uid = json.loads(ds.attrs["WB_ID_to_UID"])
    uids = sorted(list(wbid_to_uid.values()))
    uids = [i for i in uids if i not in list(multi_tile_uids.keys())]
    uids_exceptions.extend(uids)

_log.info(
    f"{len(uids_exceptions)} additional waterbodies identified for vector processing"
)

2026-03-22 13:09:11,841 __main__ [INFO]: 5378 additional waterbodies identified for vector processing


In [15]:
for uid in uids_exceptions:
    multi_tile_uids[uid] = uids_to_cog_paths[uid]

# Sort to ensure during parallel processing
# tasks requiring the shortest time are processed first.
multi_tile_uids = dict(
    sorted(multi_tile_uids.items(), key=lambda x: len(x[1]))
)

_log.info(
    f"{len(multi_tile_uids)} waterbodies in total for vector processing"
)

2026-03-22 13:09:11,851 __main__ [INFO]: 11483 waterbodies in total for vector processing


In [16]:
waterbodies_uids_fp = join_url(output_dir, "vector_processing_tasks.json")
with fs.open(waterbodies_uids_fp, "w") as file:
    json.dump(multi_tile_uids, file, indent=2)
_log.info(
    f"Waterbodies for vector processing of water quality summaries written to {waterbodies_uids_fp}"
)

2026-03-22 13:09:11,903 __main__ [INFO]: Waterbodies for vector processing of water quality summaries written to ../../src/water_quality/data/vector_processing_tasks.json


In [17]:

# Can be procesed with resource limits set to 8CPU 60GB memory 100 parallel pods
uids_2tile = {}
# Can be procesed with resource limits set to 60CPU 480GB memory 50 parallel pods
uids_3tile = {}
# Need to be resampled from 10m to higher to process using nearest neighbour
# 60CPU 480GB memory 50 parallel pods
uids_4tile_plus = {}

for uid, cog_paths in multi_tile_uids.items():
    if len(cog_paths) < 3:
        uids_2tile[uid] = cog_paths
    else:
        if len(cog_paths) == 3:
            uids_3tile[uid] = cog_paths
        else:
            uids_4tile_plus[uid] = cog_paths


In [18]:
waterbodies_uids_fp = join_url(
    output_dir, "vector_processing_tasks_2tile.json"
)
with fs.open(waterbodies_uids_fp, "w") as file:
    json.dump(uids_2tile, file, indent=2)

_log.info(
    "Waterbodies covered by 2 or less historical extent COGs for vector processing of "
    f"water quality summaries written to {waterbodies_uids_fp}"
)

waterbodies_uids_fp = join_url(
    output_dir, "vector_processing_tasks_3tile.json"
)
with fs.open(waterbodies_uids_fp, "w") as file:
    json.dump(uids_3tile, file, indent=2)

_log.info(
    "Waterbodies covered by 3 historical extent COGs for vector processing of water quality "
    f"summaries written to {waterbodies_uids_fp}"
)

waterbodies_uids_fp = join_url(
    output_dir, "vector_processing_tasks_4_plustile.json"
)
with fs.open(waterbodies_uids_fp, "w") as file:
    json.dump(uids_4tile_plus, file, indent=2)

_log.info(
    "Waterbodies covered by 4 or more historical extent COGs for vector processing of water "
    f"quality summaries written to {waterbodies_uids_fp}"
)

2026-03-22 13:09:11,959 __main__ [INFO]: Waterbodies covered by 2 or less historical extent COGs for vector processing of water quality summaries written to ../../src/water_quality/data/vector_processing_tasks_2tile.json
2026-03-22 13:09:11,961 __main__ [INFO]: Waterbodies covered by 3 historical extent COGs for vector processing of water quality summaries written to ../../src/water_quality/data/vector_processing_tasks_3tile.json
2026-03-22 13:09:11,962 __main__ [INFO]: Waterbodies covered by 4 or more historical extent COGs for vector processing of water quality summaries written to ../../src/water_quality/data/vector_processing_tasks_4_plustile.json
